In [1]:
pip install pandas openpyxl nltk rouge-score bert-score


Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from bert_score import score

# Download NLTK data if not already
nltk.download('punkt')

# --- Load your file ---
input_file = "GPT4.1_NSD.csv"  # or "your_file.xlsx"
if input_file.endswith('.csv'):
    df = pd.read_csv(input_file)
elif input_file.endswith('.xlsx'):
    df = pd.read_excel(input_file)
else:
    raise ValueError("Unsupported file type. Use .csv or .xlsx")

# Prepare ROUGE scorer
rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
smooth_fn = SmoothingFunction().method1

# For BERTScore, collect all candidates and references first
candidates = df["Model Answer"].astype(str).tolist()
references = df["Reference Answer"].astype(str).tolist()

# --- Run BERTScore once on all pairs (batch-friendly) ---
P, R, F1 = score(candidates, references, lang="en", verbose=True)

# Store results
bleu_scores = []
rouge_l_scores = []
bertscore_f1 = []

for idx, (ref, hyp) in enumerate(zip(references, candidates)):
    # Tokenize for BLEU
    ref_tokens = nltk.word_tokenize(ref)
    hyp_tokens = nltk.word_tokenize(hyp)

    # BLEU with smoothing
    bleu = sentence_bleu([ref_tokens], hyp_tokens, smoothing_function=smooth_fn)
    bleu_scores.append(bleu)

    # ROUGE-L
    rouge_score = rouge.score(ref, hyp)['rougeL'].fmeasure
    rouge_l_scores.append(rouge_score)

    # BERTScore (already computed in batch)
    bertscore_f1.append(F1[idx].item())

# Add to DataFrame
df["BLEU"] = bleu_scores
df["ROUGE-L"] = rouge_l_scores
df["BERTScore"] = bertscore_f1

# --- Save output ---
output_file = "evaluation_output.xlsx"
df.to_excel(output_file, index=False)

print(f"Done! Scores saved to {output_file}")


[nltk_data] Downloading package punkt to /Users/aijiazhou/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/17 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/17 [00:00<?, ?it/s]

done in 2633.19 seconds, 0.40 sentences/sec
Done! Scores saved to evaluation_output.xlsx
